<a href="https://colab.research.google.com/github/MuhamadRaihan356/tugas-ACO-TSP/blob/main/ACO_TSP_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ant Colony Optimization (ACO) — Traveling Salesman Problem

**Tugas:** Menyelesaikan TSP menggunakan Ant Colony Optimization  
**Graph:** Sesuai Gambar 1 (9 node: #, A, B, C, D, E, F, G, H)  
**Route:** Titik **H → D**, wajib melewati **#**

---


## 📦 Cell 1 — Install & Import Library

In [ ]:
import numpy as np
import random

print("✅ Semua library berhasil di-import!")


✅ Semua library berhasil di-import!


## 🗺️ Cell 2 — Definisi Graph

Memasukkan semua edge dan bobotnya sesuai gambar graph.  
Node index: `# = 0, A = 1, B = 2, C = 3, D = 4, E = 5, F = 6, G = 7, H = 8`


In [ ]:
# ─── Nama Node ───────────────────────────────────────────────
NODE_NAMES = ['#', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
N = len(NODE_NAMES)  # Total 9 node

# ─── Matriks Jarak ───────────────────────────────────────────
INF = float('inf')
dist = np.full((N, N), INF)
np.fill_diagonal(dist, 0)

def add_edge(i, j, w):
    dist[i][j] = w
    dist[j][i] = w

# Masukkan semua edge sesuai gambar
# Index: #=0, A=1, B=2, C=3, D=4, E=5, F=6, G=7, H=8
add_edge(0, 1, 3)   # # ─ A : 3
add_edge(0, 3, 2)   # # ─ C : 2
add_edge(0, 7, 5)   # # ─ G : 5
add_edge(1, 2, 9)   # A ─ B : 9
add_edge(1, 3, 6)   # A ─ C : 6
add_edge(2, 3, 8)   # B ─ C : 8
add_edge(2, 8, 8)   # B ─ H : 8
add_edge(3, 4, 4)   # C ─ D : 4
add_edge(3, 6, 4)   # C ─ F : 4
add_edge(4, 5, 7)   # D ─ E : 7
add_edge(4, 8, 9)   # D ─ H : 9
add_edge(5, 6, 2)   # E ─ F : 2
add_edge(5, 7, 1)   # E ─ G : 1
add_edge(5, 8, 1)   # E ─ H : 1
add_edge(7, 8, 3)   # G ─ H : 3

# Tampilkan tabel edge
print("✅ Graph berhasil didefinisikan!")
print(f"   Jumlah node : {N}")
print(f"   Node        : {NODE_NAMES}")
print()
print(f"{'Edge':<12} {'Bobot':>6}")
print("-" * 20)
edges_list = [(i,j,int(dist[i][j])) for i in range(N) for j in range(i+1,N) if dist[i][j] != INF]
for i,j,w in edges_list:
    print(f"  {NODE_NAMES[i]} ─ {NODE_NAMES[j]:<8} {w:>4}")


✅ Graph berhasil didefinisikan!
   Jumlah node : 9
   Node        : ['#', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

Edge          Bobot
--------------------
  # ─ A           3
  # ─ C           2
  # ─ G           5
  A ─ B           9
  A ─ C           6
  B ─ C           8
  B ─ H           8
  C ─ D           4
  C ─ F           4
  D ─ E           7
  D ─ H           9
  E ─ F           2
  E ─ G           1
  E ─ H           1
  G ─ H           3


## ⚙️ Cell 3 — Parameter ACO

| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| NUM_ANTS  | 20    | Jumlah semut per iterasi |
| NUM_ITER  | 100   | Total iterasi |
| ALPHA (α) | 1.0   | Bobot feromon |
| BETA  (β) | 2.0   | Bobot heuristik (1/jarak) |
| RHO   (ρ) | 0.5   | Evaporation rate feromon |
| Q         | 100   | Konstanta deposit feromon |


In [ ]:
# ─── Parameter ACO ───────────────────────────────────────────
NUM_ANTS  = 20       # Jumlah semut
NUM_ITER  = 100      # Jumlah iterasi
ALPHA     = 1.0      # Bobot feromon (tau)
BETA      = 2.0      # Bobot heuristik (eta = 1/dist)
RHO       = 0.5      # Laju evaporasi feromon
Q         = 100      # Konstanta deposit feromon
TAU_INIT  = 1.0      # Nilai awal feromon

# ─── Node Start / End / Mandatory ────────────────────────────
START = 8   # H
END   = 4   # D
MUST  = 0   # # (wajib dilewati)

# ─── Inisialisasi Matriks Feromon & Heuristik ─────────────────
tau = np.full((N, N), TAU_INIT)   # Feromon awal seragam

# Eta = 1/dist (0 jika tidak ada edge)
with np.errstate(divide='ignore', invalid='ignore'):
    eta = np.where((dist == INF) | (dist == 0), 0, 1.0 / dist)

print("✅ Parameter ACO berhasil diset!")
print(f"   Start  : {NODE_NAMES[START]} (index {START})")
print(f"   End    : {NODE_NAMES[END]} (index {END})")
print(f"   Wajib  : {NODE_NAMES[MUST]} (index {MUST})")
print()
print(f"   NUM_ANTS = {NUM_ANTS}")
print(f"   NUM_ITER = {NUM_ITER}")
print(f"   ALPHA    = {ALPHA}")
print(f"   BETA     = {BETA}")
print(f"   RHO      = {RHO}")
print(f"   Q        = {Q}")


✅ Parameter ACO berhasil diset!
   Start  : H (index 8)
   End    : D (index 4)
   Wajib  : # (index 0)

   NUM_ANTS = 20
   NUM_ITER = 100
   ALPHA    = 1.0
   BETA     = 2.0
   RHO      = 0.5
   Q        = 100


## 🔧 Cell 4 — Fungsi-Fungsi ACO

Tiga fungsi utama:
1. `get_neighbors()` → cari tetangga yang bisa dikunjungi
2. `choose_next()` → pilih node berikutnya pakai probabilitas ACO
3. `construct_path()` → bangun path semut dari H → # → D
4. `update_pheromone()` → evaporasi + deposit feromon


In [ ]:
def get_neighbors(node):
    """Kembalikan list node yang terhubung langsung dengan node ini."""
    return [j for j in range(N) if dist[node][j] != INF and dist[node][j] != 0]


def choose_next(current, visited):
    """
    Pilih node berikutnya menggunakan probabilitas ACO:
        P(i→j) = [tau(i,j)^alpha * eta(i,j)^beta] / sum(...)
    """
    neighbors = get_neighbors(current)
    candidates = [j for j in neighbors if j not in visited]

    if not candidates:
        return None

    # Hitung nilai untuk tiap kandidat
    scores = []
    for j in candidates:
        score = (tau[current][j] ** ALPHA) * (eta[current][j] ** BETA)
        scores.append(score)

    total = sum(scores)
    if total == 0:
        return random.choice(candidates)

    # Normalisasi → probabilitas
    probs = [s / total for s in scores]

    # Roulette Wheel Selection
    r = random.random()
    cumulative = 0.0
    for idx, j in enumerate(candidates):
        cumulative += probs[idx]
        if r <= cumulative:
            return j
    return candidates[-1]


def construct_path(start, end, mandatory):
    """
    Bangun path: start → ... → mandatory → ... → end
    Dibagi 2 fase agar constraint mandatory terpenuhi.
    """
    # ── FASE 1: start → mandatory ──────────────────────────────
    path1  = [start]
    visited = {start}
    current = start
    found   = False

    for _ in range(N * 3):
        if current == mandatory:
            found = True
            break
        nxt = choose_next(current, visited)
        if nxt is None:
            break
        path1.append(nxt)
        visited.add(nxt)
        current = nxt

    if not found:
        return None, INF

    # ── FASE 2: mandatory → end ────────────────────────────────
    path2   = [mandatory]
    visited2 = visited.copy()
    current  = mandatory
    found2   = False

    for _ in range(N * 3):
        if current == end:
            found2 = True
            break
        nxt = choose_next(current, visited2)
        if nxt is None:
            break
        path2.append(nxt)
        visited2.add(nxt)
        current = nxt

    if not found2:
        return None, INF

    # ── Gabung path ────────────────────────────────────────────
    full_path = path1 + path2[1:]   # hindari duplikasi node mandatory

    # Hitung total jarak
    total_dist = 0
    for i in range(len(full_path) - 1):
        d = dist[full_path[i]][full_path[i+1]]
        if d == INF:
            return None, INF
        total_dist += d

    return full_path, total_dist


def update_pheromone(all_paths, all_costs):
    """
    Update matriks feromon:
    1. Evaporasi : tau = tau * (1 - rho)
    2. Deposit   : tau += Q / cost  (untuk setiap semut yang berhasil)
    """
    global tau
    tau *= (1 - RHO)   # Evaporasi

    for path, cost in zip(all_paths, all_costs):
        if path is not None and cost < INF:
            deposit = Q / cost
            for i in range(len(path) - 1):
                tau[path[i]][path[i+1]] += deposit
                tau[path[i+1]][path[i]] += deposit

    tau = np.clip(tau, 0.01, None)  # Feromon minimum = 0.01


print("✅ Semua fungsi ACO berhasil didefinisikan!")
print("   - get_neighbors()")
print("   - choose_next()")
print("   - construct_path()")
print("   - update_pheromone()")


✅ Semua fungsi ACO berhasil didefinisikan!
   - get_neighbors()
   - choose_next()
   - construct_path()
   - update_pheromone()


## 🚀 Cell 5 — Jalankan ACO (Main Loop)

Setiap iterasi:
- 20 semut membangun path masing-masing
- Path terbaik dicatat
- Feromon di-update (evaporasi + deposit)


In [ ]:
# Reset feromon sebelum mulai
tau = np.full((N, N), TAU_INIT)

best_path = None
best_cost = INF
history_best = []
history_avg  = []

print("=" * 55)
print("   MENJALANKAN ANT COLONY OPTIMIZATION")
print("=" * 55)
print(f"  Start : {NODE_NAMES[START]}  |  End : {NODE_NAMES[END]}  |  Wajib : {NODE_NAMES[MUST]}")
print("-" * 55)

for iteration in range(NUM_ITER):
    iter_paths = []
    iter_costs = []

    # Setiap semut bangun path
    for ant in range(NUM_ANTS):
        path, cost = construct_path(START, END, MUST)
        iter_paths.append(path)
        iter_costs.append(cost)

        # Update global best
        if path is not None and cost < best_cost:
            best_cost = cost
            best_path = path[:]

    # Update feromon
    update_pheromone(iter_paths, iter_costs)

    # Catat statistik
    valid = [c for c in iter_costs if c < INF]
    avg   = np.mean(valid) if valid else best_cost
    history_best.append(best_cost)
    history_avg.append(avg)

    # Print setiap 10 iterasi
    if (iteration + 1) % 10 == 0:
        p_str = ' → '.join([NODE_NAMES[n] for n in best_path]) if best_path else 'Belum ditemukan'
        print(f"  Iter {iteration+1:3d} | Best: {best_cost:.0f} | Avg: {avg:.1f} | {p_str}")

print("=" * 55)
print("✅ ACO selesai!")


   MENJALANKAN ANT COLONY OPTIMIZATION
  Start : H  |  End : D  |  Wajib : #
-------------------------------------------------------
  Iter  10 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  20 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  30 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  40 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  50 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  60 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  70 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  80 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter  90 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
  Iter 100 | Best: 13 | Avg: 13.0 | H → E → G → # → C → D
✅ ACO selesai!


## 🏆 Cell 6 — Tampilkan Hasil Terbaik

In [ ]:
print()
print("╔══════════════════════════════════════════════╗")
print("║         HASIL TERBAIK ACO                   ║")
print("╠══════════════════════════════════════════════╣")

if best_path:
    path_str = ' → '.join([NODE_NAMES[n] for n in best_path])
    print(f"║  Path   : {path_str}")
    print(f"║  Jarak  : {int(best_cost)}")
    print(f"║  Node   : {len(best_path)} titik")
    print("╠══════════════════════════════════════════════╣")
    print("║  Detail Setiap Edge:                        ║")
    total = 0
    for i in range(len(best_path) - 1):
        u, v = best_path[i], best_path[i+1]
        w = int(dist[u][v])
        total += w
        print(f"║    {NODE_NAMES[u]:>2} → {NODE_NAMES[v]:<2}  :  jarak = {w}")
    print(f"║    {'─'*30}")
    print(f"║    Total jarak = {total}")
    print("╚══════════════════════════════════════════════╝")
else:
    print("║  ❌ Tidak ditemukan path yang valid!         ║")
    print("╚══════════════════════════════════════════════╝")



╔══════════════════════════════════════════════╗
║         HASIL TERBAIK ACO                   ║
╠══════════════════════════════════════════════╣
║  Path   : H → E → G → # → C → D
║  Jarak  : 13
║  Node   : 6 titik
╠══════════════════════════════════════════════╣
║  Detail Setiap Edge:                        ║
║     H → E   :  jarak = 1
║     E → G   :  jarak = 1
║     G → #   :  jarak = 5
║     # → C   :  jarak = 2
║     C → D   :  jarak = 4
║    ──────────────────────────────
║    Total jarak = 13
╚══════════════════════════════════════════════╝


## 📊 Cell 7 — Tampilkan Semua Hasil (Teks)

In [ ]:

SEP  = '=' * 55
SEP2 = '-' * 55

# ── 1. Ringkasan Graph ───────────────────────────────────────
print(SEP)
print('  RINGKASAN GRAPH')
print(SEP)
print(f'  Node  : {NODE_NAMES}')
print(f'  Start : {NODE_NAMES[START]}  (index {START})')
print(f'  End   : {NODE_NAMES[END]}  (index {END})')
print(f'  Wajib : {NODE_NAMES[MUST]}  (index {MUST})')
print()
print(f'  {"Edge":<10} {"Bobot":>6}')
print('  ' + SEP2[:30])
edges_all = [(i, j, int(dist[i][j]))
             for i in range(N)
             for j in range(i+1, N)
             if dist[i][j] != INF]
for i, j, w in edges_all:
    marker = ' <-- JALUR TERBAIK' if best_path and (
        any(best_path[k]==i and best_path[k+1]==j or
            best_path[k]==j and best_path[k+1]==i
            for k in range(len(best_path)-1))) else ''
    print(f'  {NODE_NAMES[i]} -- {NODE_NAMES[j]:<6} {w:>4}{marker}')

# ── 2. Proses Iterasi ────────────────────────────────────────
print()
print(SEP)
print('  PROSES ITERASI ACO')
print(SEP)
print(f'  {"Iterasi":<10} {"Best Cost":>10} {"Avg Cost":>10}')
print('  ' + SEP2)
step = max(1, NUM_ITER // 10)
for i in range(step-1, NUM_ITER, step):
    avg_val = f'{history_avg[i]:.1f}' if history_avg[i] < float("inf") else 'inf'
    print(f'  {i+1:<10} {history_best[i]:>10.1f} {avg_val:>10}')

# ── 3. Jalur Terbaik ─────────────────────────────────────────
print()
print(SEP)
print('  HASIL TERBAIK ACO')
print(SEP)
if best_path:
    path_str = ' -> '.join([NODE_NAMES[n] for n in best_path])
    print(f'  Jalur        : {path_str}')
    print(f'  Total Jarak  : {int(best_cost)}')
    print(f'  Jumlah Node  : {len(best_path)}')
    print()
    print(f'  {"No":<5} {"Dari":<6} {"Ke":<6} {"Jarak":>6}')
    print('  ' + SEP2)
    running = 0
    for k in range(len(best_path) - 1):
        u, v = best_path[k], best_path[k+1]
        w = int(dist[u][v])
        running += w
        print(f'  {k+1:<5} {NODE_NAMES[u]:<6} {NODE_NAMES[v]:<6} {w:>6}')
    print('  ' + SEP2)
    print(f'  {"":<5} {"":<6} {"TOTAL":<6} {running:>6}')
else:
    print('  Tidak ditemukan path yang valid!')

# ── 4. Tabel Feromon Akhir ───────────────────────────────────
print()
print(SEP)
print('  FEROMON AKHIR (hanya edge yang ada)')
print(SEP)
print(f'  {"Edge":<10} {"Feromon":>10}')
print('  ' + SEP2[:30])
for i, j, _ in edges_all:
    print(f'  {NODE_NAMES[i]} -- {NODE_NAMES[j]:<6} {tau[i][j]:>10.4f}')

# ── 5. Konvergensi (Teks) ────────────────────────────────────
print()
print(SEP)
print('  GRAFIK KONVERGENSI (TEXT BAR CHART)')
print(SEP)
print(f'  Best cost awal  : {history_best[0]:.0f}')
print(f'  Best cost akhir : {history_best[-1]:.0f}')
print()
max_cost = max(c for c in history_best if c < float('inf'))
bar_width = 30
print(f'  {"Iter":<6} Bar (Best Cost)')
print('  ' + SEP2)
for i in range(0, NUM_ITER, step):
    cost = history_best[i]
    if cost < float('inf'):
        filled = int((cost / max_cost) * bar_width)
        bar = '#' * filled + '.' * (bar_width - filled)
        print(f'  {i+1:<6} [{bar}] {cost:.0f}')

print()
print(SEP)
print('  SELESAI')
print(SEP)


  RINGKASAN GRAPH
  Node  : ['#', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
  Start : H  (index 8)
  End   : D  (index 4)
  Wajib : #  (index 0)

  Edge        Bobot
  ------------------------------
  # -- A         3
  # -- C         2 <-- JALUR TERBAIK
  # -- G         5 <-- JALUR TERBAIK
  A -- B         9
  A -- C         6
  B -- C         8
  B -- H         8
  C -- D         4 <-- JALUR TERBAIK
  C -- F         4
  D -- E         7
  D -- H         9
  E -- F         2
  E -- G         1 <-- JALUR TERBAIK
  E -- H         1 <-- JALUR TERBAIK
  G -- H         3

  PROSES ITERASI ACO
  Iterasi     Best Cost   Avg Cost
  -------------------------------------------------------
  10               13.0       13.0
  20               13.0       13.0
  30               13.0       13.0
  40               13.0       13.0
  50               13.0       13.0
  60               13.0       13.0
  70               13.0       13.0
  80               13.0       13.0
  90               13.0       13.

## 📝 Kesimpulan

### Cara Kerja ACO pada TSP ini:

1. **Inisialisasi** — Semua edge diberi feromon awal yang sama (τ = 1.0)
2. **Konstruksi Path** — Setiap semut membangun jalur dari H → # → D menggunakan probabilitas:

$$P(i \to j) = \frac{[\tau_{ij}]^\alpha \cdot [\eta_{ij}]^\beta}{\sum_{k \in \text{candidates}} [\tau_{ik}]^\alpha \cdot [\eta_{ik}]^\beta}$$

3. **Update Feromon** — Evaporasi (τ × (1-ρ)) lalu deposit (Q/cost) pada jalur yang dilalui
4. **Iterasi** — Proses diulang 100× hingga konvergen ke jalur terbaik

### Hasil:
| | |
|---|---|
| **Jalur Terbaik** | H → E → G → # → C → D |
| **Total Jarak** | 13 |
| **Constraint** | ✅ Melewati node # |
